# Feasibility test: Qwen-Image-Layered (20B) on a free Kaggle T4

Same code as the local dry run (`layerproto/qwen_layered.py`), with the real weights:
8-bit text encoder (stage 1, then unloaded) and the 4-bit GGUF transformer (stage 2), one T4 at a time.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "diffusers==0.37.0" "transformers>=4.51" accelerate gguf
import importlib.metadata as m
print({p: m.version(p) for p in ['torch', 'diffusers', 'transformers', 'gguf', 'accelerate']})

In [ ]:
import os, sys
WORK = "/kaggle/working"
os.makedirs(WORK + '/layerproto', exist_ok=True)
open(WORK + '/layerproto/__init__.py', 'w').close()
open(WORK + '/layerproto/qwen_layered.py', 'w', encoding='utf-8').write("\"\"\"Qwen-Image-Layered decomposition: one finished image -> N RGBA layers that composite back to it.\n\nShared by the local dry run (tools/dryrun_qwen_layered.py, tiny random models, CPU) and the Kaggle test notebook\n(real 8-bit text encoder + 4-bit GGUF transformer on a T4), so the call path is verified before any big download.\n\nTwo stages so a 16 GB T4 is enough:\n    1. text encoder encodes the caption (prompt embeddings), then is unloaded\n    2. transformer decomposes the image using those embeddings\n\"\"\"\nfrom __future__ import annotations\n\nimport gc\nimport time\nimport types\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\nREPO = \"Qwen/Qwen-Image-Layered\"\n\n\ndef make_pipeline(text_encoder, vae, repo: str = REPO):\n    \"\"\"Pipeline without a transformer yet (it is loaded after the text encoder is gone).\"\"\"\n    from diffusers import FlowMatchEulerDiscreteScheduler, QwenImageLayeredPipeline\n    from transformers import Qwen2Tokenizer, Qwen2VLProcessor\n\n    return QwenImageLayeredPipeline(\n        scheduler=FlowMatchEulerDiscreteScheduler.from_pretrained(repo, subfolder=\"scheduler\"),\n        vae=vae, text_encoder=text_encoder,\n        tokenizer=Qwen2Tokenizer.from_pretrained(repo, subfolder=\"tokenizer\"),\n        processor=Qwen2VLProcessor.from_pretrained(repo, subfolder=\"processor\"),\n        transformer=None)\n\n\ndef encode(pipe, caption: str, device, max_len: int = 512):\n    \"\"\"Prompt + negative (' ') embeddings. encode_prompt returns mask=None when the mask is all ones, but\n    __call__ insists on a mask whenever embeddings are passed in, so all-ones masks are rebuilt here.\"\"\"\n    with torch.no_grad():\n        pe, pm = pipe.encode_prompt(prompt=caption, device=device, num_images_per_prompt=1, max_sequence_length=max_len)\n        ne, nm = pipe.encode_prompt(prompt=\" \", device=device, num_images_per_prompt=1, max_sequence_length=max_len)\n\n    def ones(e):\n        return torch.ones(e.shape[:2], dtype=torch.long, device=e.device)\n\n    return pe, (pm if pm is not None else ones(pe)), ne, (nm if nm is not None else ones(ne))\n\n\ndef cached_encode(pipe, caption: str, device, cache_dir: str, max_len: int = 512):\n    \"\"\"encode() with a disk cache keyed by caption: a cache hit means the 15 GB text encoder never has to be loaded\n    (Kaggle: ~70 s load + RAM). Typical loop use: one fixed caption (\"\" or the scene prompt) per scene.\"\"\"\n    import hashlib\n    import os\n    os.makedirs(cache_dir, exist_ok=True)\n    f = os.path.join(cache_dir, hashlib.sha1(f\"{caption}|{max_len}\".encode()).hexdigest()[:16] + \".pt\")\n    if os.path.exists(f):\n        return tuple(t.to(device) for t in torch.load(f, map_location=\"cpu\"))\n    out = encode(pipe, caption, device, max_len)\n    torch.save(tuple(t.detach().cpu() for t in out), f)\n    return out\n\n\nclass HeunVelocity:\n    \"\"\"Turn the pipeline's Euler step into Heun's method (what Stable-Layers' decompose.py uses) without touching the\n    pipeline: the transformer wrapper returns v_heun = (v(x, s) + v(x + dt*v(x, s), s_next)) / 2, so the pipeline's\n    own Euler update x + dt*v_heun is exactly a Heun step. Final step (s_next = 0) stays Euler, as in Stable-Layers.\n    Only the generated tokens are advanced for the midpoint; the trailing condition-image tokens stay fixed.\n    Cost: 2 transformer calls per step (1 on the last step).\"\"\"\n\n    def __init__(self, transformer, scheduler, layers: int):\n        self.t, self.sched, self.frac = transformer, scheduler, (layers + 1) / (layers + 2)\n        self.orig = transformer.forward\n\n    def __enter__(self):\n        self.t.forward = self.forward\n        return self\n\n    def __exit__(self, *a):\n        self.t.forward = self.orig\n\n    def _next_sigma(self, s):\n        sig = self.sched.sigmas.to(torch.float32).cpu()\n        i = int(torch.argmin((sig - s).abs()))\n        return float(sig[i + 1]) if i + 1 < len(sig) else 0.0\n\n    def forward(self, hidden_states, *a, timestep=None, **kw):\n        out1 = self.orig(hidden_states, *a, timestep=timestep, **kw)\n        v1 = out1[0] if isinstance(out1, tuple) else out1.sample\n        s = float(timestep.flatten()[0])                 # pipeline passes sigma (timestep / 1000)\n        s_next = self._next_sigma(s)\n        if s_next <= 0:\n            return out1\n        n = int(round(hidden_states.shape[1] * self.frac))\n        x_mid = hidden_states.clone()\n        x_mid[:, :n] = hidden_states[:, :n] + (s_next - s) * v1[:, :n].to(hidden_states.dtype)\n        out2 = self.orig(x_mid, *a, timestep=torch.full_like(timestep, s_next), **kw)\n        v2 = out2[0] if isinstance(out2, tuple) else out2.sample\n        v = 0.5 * (v1 + v2)\n        if isinstance(out1, tuple):\n            return (v,) + tuple(out1[1:])\n        out1.sample = v\n        return out1\n\n\ndef unload_text_encoder(pipe, dtype):\n    \"\"\"Free the text encoder. The pipeline still reads text_encoder.dtype later, so leave a stand-in;\n    object.__setattr__ bypasses diffusers' component registration (which rejects non-modules).\"\"\"\n    te = pipe.text_encoder\n    object.__setattr__(pipe, \"text_encoder\", types.SimpleNamespace(dtype=dtype))\n    del te\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef fix_gguf_non_linear(model, dtype) -> int:\n    \"\"\"diffusers' GGUF loader only swaps nn.Linear for dequantizing GGUFLinear layers. Any other module keeps its raw\n    packed bytes, e.g. the layered model's addition_t_embedding (nn.Embedding(2, 3072)) returns 6144 bytes instead of\n    3072 fp16 values -> 'size of tensor a (3072) must match ... (6144)'. Dequantize those parameters once after loading.\"\"\"\n    from diffusers.quantizers.gguf.utils import GGUFLinear, GGUFParameter, dequantize_gguf_tensor\n\n    fixed = 0\n    for mod in model.modules():\n        for name, p in list(mod.named_parameters(recurse=False)):\n            if isinstance(mod, GGUFLinear) and name == \"weight\":\n                continue  # GGUFLinear dequantizes its own weight; its bias is not, so it is handled here too\n            if isinstance(p, GGUFParameter):\n                setattr(mod, name, torch.nn.Parameter(dequantize_gguf_tensor(p).to(dtype), requires_grad=False))\n                fixed += 1\n    return fixed\n\n\ndef decompose(pipe, transformer, image: Image.Image, caption: str, embeds, steps: int = 50, layers: int = 4,\n              resolution: int = 640, seed: int = 777, true_cfg: float = 4.0, on_step=None,\n              offload: bool = True, sampler: str = \"euler\"):\n    \"\"\"Run the decomposition. Returns (list of RGBA layers, stats). sampler=\"heun\" doubles the transformer calls\n    (Stable-Layers recommends Heun; its decompose.py uses linear sigmas, here the pipeline's schedule is kept).\"\"\"\n    pipe.register_modules(transformer=transformer)\n    image = image.convert(\"RGBA\")  # the layered VAE has 4 input channels (as in the model card example)\n    pe, pm, ne, nm = embeds\n    gen_device = \"cuda\" if pe.is_cuda else \"cpu\"\n    times, grabbed = [], {}\n\n    class _Stop(Exception):\n        pass\n\n    def cb(p, i, t, kw):\n        times.append(time.time())\n        if on_step:\n            on_step(i)\n        if i == len(p.scheduler.timesteps) - 1:   # last step (Heun schedulers have ~2x steps timesteps)\n            # Take the final latents and stop before the pipeline's own decode: it would run the VAE while the\n            # 12.7 GB transformer is still on the GPU (OOM on a 15 GB T4), and output_type=\"latent\" is broken in\n            # diffusers 0.37 (that branch returns an unassigned `images` variable).\n            grabbed[\"latents\"] = kw[\"latents\"]\n            raise _Stop\n        return kw\n\n    # the pipeline only accepts 640 / 1024; for other sizes (e.g. 512, 384) swap in our target area\n    import diffusers.pipelines.qwenimage.pipeline_qwenimage_layered as _pl\n    _orig_cd = _pl.calculate_dimensions\n    if resolution not in (640, 1024):\n        _pl.calculate_dimensions = lambda area, ratio: _orig_cd(resolution * resolution, ratio)\n    t0 = time.time()\n    import contextlib\n    heun = HeunVelocity(transformer, pipe.scheduler, layers) if sampler == \"heun\" else contextlib.nullcontext()\n    try:\n        with torch.no_grad(), heun:\n            pipe(image=image, prompt=caption, prompt_embeds=pe, prompt_embeds_mask=pm,\n                 negative_prompt_embeds=ne, negative_prompt_embeds_mask=nm, true_cfg_scale=true_cfg,\n                 layers=layers, resolution=resolution if resolution in (640, 1024) else 640, num_inference_steps=steps, cfg_normalize=True,\n                 generator=torch.Generator(gen_device).manual_seed(seed), callback_on_step_end=cb,\n                 callback_on_step_end_tensor_inputs=[\"latents\"])\n    except _Stop:\n        pass\n    finally:\n        _pl.calculate_dimensions = _orig_cd\n    stats = {\"denoise_s\": round(time.time() - t0, 1)}\n    if len(times) > 1:\n        stats[\"s_per_step\"] = round((times[-1] - times[0]) / (len(times) - 1), 2)\n\n    # free the GPU before decoding: move the weights to CPU RAM (works even while the caller still holds `transformer`).\n    # offload=False keeps it on the GPU(s) (enough room when it is split over two T4s or at small resolutions)\n    if offload:\n        transformer.to(\"cpu\")\n    pipe.register_modules(transformer=None)\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n    t1 = time.time()\n    imgs = decode_latents(pipe, grabbed[\"latents\"], image.size, layers, resolution)\n    stats[\"decode_s\"] = round(time.time() - t1, 1)\n    return imgs, stats\n\n\ndef decode_latents(pipe, latents, image_size, layers: int, resolution: int):\n    \"\"\"Same maths as the end of QwenImageLayeredPipeline.__call__, but decoding one layer at a time.\"\"\"\n    from diffusers.pipelines.qwenimage.pipeline_qwenimage_layered import calculate_dimensions\n\n    w, h = calculate_dimensions(resolution * resolution, image_size[0] / image_size[1])\n    m = pipe.vae_scale_factor * 2\n    w, h = w // m * m, h // m * m\n    vae = pipe.vae\n    lat = pipe._unpack_latents(latents, h, w, layers, pipe.vae_scale_factor).to(vae.dtype)\n    mean = torch.tensor(vae.config.latents_mean).view(1, vae.config.z_dim, 1, 1, 1).to(lat.device, lat.dtype)\n    inv_std = 1.0 / torch.tensor(vae.config.latents_std).view(1, vae.config.z_dim, 1, 1, 1).to(lat.device, lat.dtype)\n    lat = lat / inv_std + mean\n    b, c, f, hh, ww = lat.shape\n    lat = lat[:, :, 1:]  # first frame is the input image, not an output layer\n    lat = lat.permute(0, 2, 1, 3, 4).reshape(-1, c, 1, hh, ww)\n    out = []\n    with torch.no_grad():\n        for k in range(lat.shape[0]):\n            img = vae.decode(lat[k:k + 1], return_dict=False)[0].squeeze(2)\n            out.extend(pipe.image_processor.postprocess(img, output_type=\"pil\"))\n            del img\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n    return [im.convert(\"RGBA\") for im in out]\n\n\ndef recompose(layers, size=None):\n    \"\"\"Stack the layers bottom to top; returns (image, per-layer alpha stats).\"\"\"\n    canvas, info = None, []\n    for L in layers:\n        a = np.array(L)[..., 3]\n        info.append({\"size\": L.size, \"transparent_pct\": round(100 * float((a < 10).mean()), 1),\n                     \"opaque_pct\": round(100 * float((a > 245).mean()), 1),\n                     \"has_nan_black\": bool((np.array(L)[..., :3].max(-1) == 0).mean() > 0.95)})\n        canvas = L if canvas is None else Image.alpha_composite(canvas, L.resize(canvas.size))\n    if canvas is not None and size:\n        canvas = canvas.resize(size)\n    return canvas, info\n\n\ndef strip(images, w: int = 256) -> Image.Image:\n    \"\"\"Side-by-side preview; green shows transparency.\"\"\"\n    out = Image.new(\"RGBA\", (w * len(images), w), (40, 160, 60, 255))\n    for i, im in enumerate(images):\n        tile = Image.new(\"RGBA\", (w, w), (40, 160, 60, 255))\n        tile.alpha_composite(im.convert(\"RGBA\").resize((w, w)))\n        out.paste(tile, (i * w, 0))\n    return out\n")
open(WORK + '/layerproto/fp16_mixed.py', 'w', encoding='utf-8').write("\"\"\"Run the Qwen-Image(-Layered) transformer on GPUs without bf16 (Kaggle T4, P100, RTX 20xx) without NaNs.\n\nQwen-Image is trained for bf16; its hidden values exceed the fp16 limit (65504), so a plain fp16 run gives all-NaN\nlatents (ComfyUI also lists Qwen-Image as bf16/fp32 only). This keeps the fast fp16 tensor-core matmuls but holds\neverything else in fp32:\n\n  * every Linear: input scaled down to absmax 32, fp16 matmul, result scaled back in fp32\n    (falls back to a fp32 matmul for that call if the fp16 result is still non-finite)\n  * attention: q/k in fp16 (RMS-normalised, small), v scaled the same way (attention is linear in v)\n  * residual stream, norms, gates, embeddings: fp32\n  * the model's output is returned in fp16, so the diffusers pipeline is unchanged\n\nMeasured on Kaggle 2x T4 (Q4_K_S GGUF, 640 px, 4 layers): no NaN, 42 s/step, peak 8.8 / 8.4 GB.\n\n    from layerproto.fp16_mixed import apply_fp16_mixed\n    apply_fp16_mixed(transformer)        # once, after loading; idempotent\n\"\"\"\nimport types\n\nimport torch\nimport torch.nn.functional as F\n\nLINEAR_LIMIT = 32.0  # absmax the fp16 matmul input is scaled to (1024 was still too high: NaNs)\nATTN_LIMIT = 32.0\n\n\ndef _dequant(w):\n    if type(w).__name__ == \"GGUFParameter\":\n        from diffusers.quantizers.gguf.utils import dequantize_gguf_tensor\n        return dequantize_gguf_tensor(w)\n    return w\n\n\ndef _mixed_linear(self, x):\n    x = x.float()\n    s = (x.abs().amax() / LINEAR_LIMIT).clamp_(min=1.0)\n    y = self._fp16_orig_forward((x / s).to(torch.float16)).float()   # = W x/s + b\n    b = getattr(self, \"bias\", None)\n    if not torch.isfinite(y).all():                                  # rare: this call in fp32\n        y = F.linear(x, _dequant(self.weight).float(), None if b is None else b.float())\n    elif b is None:\n        y = y * s\n    else:\n        b = b.float()\n        y = (y - b) * s + b\n    loras = getattr(self, \"_fp16_lora\", None)\n    if loras:                                                        # low-rank deltas (stacked LoRAs), same input scaling\n        xs = (x / s).to(torch.float16)\n        for i, (A, B, k) in enumerate(loras):\n            if A.device != x.device:                                 # module was moved (e.g. split over GPUs)\n                A, B = A.to(x.device), B.to(x.device)\n                loras[i] = (A, B, k)\n            y = y + F.linear(F.linear(xs, A), B).float() * (s * k)\n    return y\n\n\ndef _lora_pairs(sd):\n    \"\"\"{module name: (A [r, in], B [out, r], alpha or None)} for common LoRA key layouts (PEFT / kohya / diffusers).\"\"\"\n    out = {}\n    for key in sd:\n        for a_suf, b_suf in ((\".lora_A.weight\", \".lora_B.weight\"), (\".lora_down.weight\", \".lora_up.weight\"),\n                             (\".lora_A.default.weight\", \".lora_B.default.weight\")):\n            if key.endswith(a_suf):\n                base = key[: -len(a_suf)]\n                bkey = base + b_suf\n                if bkey in sd:\n                    alpha = sd.get(base + \".alpha\")\n                    name = base\n                    for pre in (\"base_model.model.\", \"diffusion_model.\", \"transformer.\", \"model.\"):\n                        if name.startswith(pre):\n                            name = name[len(pre):]\n                    out[name] = (sd[key], sd[bkey], None if alpha is None else float(alpha))\n    return out\n\n\ndef add_lora(model, path_or_state_dict, scale: float = 1.0):\n    \"\"\"Attach a LoRA to an fp16_mixed-patched model (works with GGUF-quantised linears, nothing is merged).\n    Returns (matched modules, LoRA modules not found).\"\"\"\n    sd = path_or_state_dict\n    if isinstance(sd, str):\n        from safetensors.torch import load_file\n        sd = load_file(sd)\n    pairs = _lora_pairs(sd)\n    mods = dict(model.named_modules())\n    hit, miss = 0, []\n    for name, (A, B, alpha) in pairs.items():\n        m = mods.get(name)\n        if m is None or not hasattr(m, \"_fp16_orig_forward\"):\n            miss.append(name)\n            continue\n        dev = next((p.device for p in m.parameters()), torch.device(\"cpu\"))\n        r = A.shape[0]\n        k = scale * ((alpha / r) if alpha else 1.0)\n        # append, never replace: ART+ stacks three LoRAs on the same modules (v1-v8 silently kept only the last one)\n        if not hasattr(m, \"_fp16_lora\"):\n            m._fp16_lora = []\n        m._fp16_lora.append((A.to(dev, torch.float16), B.to(dev, torch.float16), k))\n        hit += 1\n    return hit, miss\n\n\ndef remove_lora(model):\n    for _, m in model.named_modules():\n        if hasattr(m, \"_fp16_lora\"):\n            del m._fp16_lora\n\n\ndef _patch_attention():\n    # Qwen-Image attention only; other models (FLUX ports with older bundled diffusers) just get the Linear patch\n    try:\n        import diffusers.models.transformers.transformer_qwenimage as tq\n    except ImportError:\n        return\n    if getattr(tq, \"_fp16_orig_attn\", None) is not None:\n        return\n    orig = tq.dispatch_attention_fn\n\n    def attn_fp16(q, k, v, attn_mask=None, **kw):\n        s = (v.float().abs().amax() / ATTN_LIMIT).clamp_(min=1.0)\n        if attn_mask is not None and attn_mask.is_floating_point():\n            attn_mask = attn_mask.to(torch.float16)\n        o = orig(q.to(torch.float16), k.to(torch.float16), (v.float() / s).to(torch.float16), attn_mask=attn_mask, **kw)\n        return o.float() * s\n\n    tq._fp16_orig_attn = orig\n    tq.dispatch_attention_fn = attn_fp16\n\n\ndef apply_fp16_mixed(model):\n    \"\"\"Patch a loaded QwenImageTransformer2DModel in place. Returns (linears patched, params moved to fp32).\"\"\"\n    _patch_attention()\n    n_lin = n_par = 0\n    for _, m in model.named_modules():\n        if \"Linear\" in type(m).__name__:\n            if not hasattr(m, \"_fp16_orig_forward\"):\n                m._fp16_orig_forward = m.forward\n            m.forward = types.MethodType(_mixed_linear, m)\n            n_lin += 1\n        else:\n            for _, p in m.named_parameters(recurse=False):\n                if p.dtype == torch.float16:\n                    p.data = p.data.float()\n                    n_par += 1\n    if not hasattr(model, \"_fp16_orig_model_forward\"):\n        model._fp16_orig_model_forward = model.forward\n\n        def fwd(*a, **k):\n            o = model._fp16_orig_model_forward(*a, **k)\n            if isinstance(o, tuple):\n                if isinstance(o[0], (list, tuple)):          # ControlNet-style adapters return (list of block samples, ...)\n                    return tuple([x.to(torch.float16) for x in e] if isinstance(e, (list, tuple)) else e for e in o)\n                return (o[0].to(torch.float16),) + tuple(o[1:])\n            if hasattr(o, \"sample\"):\n                o.sample = o.sample.to(torch.float16)\n            return o\n\n        model.forward = fwd\n    return n_lin, n_par\n")
sys.path.insert(0, WORK)
from layerproto import qwen_layered as Q
print('module ok')

In [ ]:
import base64, io, json, time, gc
import torch
from PIL import Image
os.environ['HF_HUB_DISABLE_XET'] = '1'
OUT = WORK + '/qwen_layered_test'; os.makedirs(OUT, exist_ok=True)
CAPTION = "An impressionist oil painting of a forest clearing at night: tall pine trees and bare trees, a full moon and stars in a blue sky, distant mountains, a campfire with logs in the middle, a wooden chair next to it, rocks and fallen branches on the ground."
img = Image.open(io.BytesIO(base64.b64decode('...'))).convert('RGB'); img.save(f'{OUT}/input.png')
EXTRA = {k: (Image.open(io.BytesIO(base64.b64decode(b))).convert('RGB'), c) for k, (b, c) in {"fox_cottage": ["...", "Storybook illustration of a thatched stone cottage with a large leafy tree on the right, a stone gate arch on the left, small village houses behind, a cobblestone path leading into the foreground and a red fox standing on the path."], "q21_night": ["...", "Storybook illustration of a thatched stone cottage with a large leafy tree on the right, a stone gate arch on the left, small village houses behind, a cobblestone path leading into the foreground and a red fox standing on the path. Now at night, moonlit, lantern light."], "q21_rain": ["...", "Storybook illustration of a thatched stone cottage with a large leafy tree on the right, a stone gate arch on the left, small village houses behind, a cobblestone path leading into the foreground and a red fox standing on the path. Now in heavy rain, grey sky."], "q21_t2i": ["...", "Storybook illustration of a thatched stone cottage with a large leafy tree on the right, a stone gate arch on the left, small village houses behind, a cobblestone path leading into the foreground and a red fox standing on the path."], "pixelart_camp": ["...", "A colourful pixel-art night scene: a skeleton sitting next to a small brick campfire, stylised white, yellow and green trees, a big yellow moon and a small orange moon in a dark blue starry sky, brown ground."], "sketch_skeleton": ["...", "A pale illustration of a skeleton sitting at a small wooden table next to a campfire with logs and stones, two cut logs, misty pine trees and a few stars, all inside a thin circle on a white background."]}.items()}
DT = torch.float16
import glob
_g = glob.glob('/kaggle/input/**/qwen-image-layered-Q4_K_S.gguf', recursive=True)
_b = glob.glob('/kaggle/input/**/Qwen-Image-Layered/model_index.json', recursive=True)
GGUF = _g[0] if _g else None                         # cached transformer (kernel output), else Hugging Face
SRC = os.path.dirname(_b[0]) if _b else Q.REPO       # cached text encoder / VAE / configs, else Hugging Face
print('transformer from:', GGUF or 'Hugging Face', '| base from:', SRC, flush=True)
stats = {'weights_cached': bool(GGUF and _b)}
def peak(tag):
    stats[tag + '_peak_GB'] = [round(torch.cuda.max_memory_allocated(i) / 1e9, 2) for i in range(torch.cuda.device_count())]
    print(tag, 'peak GB per GPU:', stats[tag + '_peak_GB'], flush=True)
def save():
    json.dump(stats, open(f'{OUT}/stats.json', 'w'), indent=1)

In [ ]:
_sh = sorted(glob.glob('/kaggle/input/**/Qwen-Image-Layered/text_encoder/*.safetensors', recursive=True))
if _sh:
    _f = max(_sh, key=os.path.getsize); _t = time.time(); _n = 0
    with open(_f, 'rb', buffering=0) as _fh:
        while time.time() - _t < 10:
            _b = _fh.read(64 * 1024 * 1024)
            if not _b:
                break
            _n += len(_b)
    stats['input_read_MBps'] = round(_n / 1e6 / max(time.time() - _t, 1e-3), 1)
    print('cached text encoder read speed:', stats['input_read_MBps'], 'MB/s', flush=True); save()
    assert stats['input_read_MBps'] > 100, 'Cached inputs read too slowly (still being copied?) - stop the session and retry later'
else:
    print('no cached text encoder attached - will download from Hugging Face')

## Stage 1: fp16 text encoder split over both T4s -> prompt embeddings, then unload

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration
from diffusers import AutoencoderKLQwenImage
import shutil
# transformers 5 loads tensors with many parallel threads; on the /kaggle/input mount those reads stalled at
# 0/729 for minutes (single sequential reads are fast). Copy the base weights to local disk once per session.
t = time.time()
LOCAL = '/tmp/Qwen-Image-Layered'
os.system('df -h /tmp | tail -n 1')
if SRC.startswith('/kaggle/input'):
    shutil.copytree(SRC, LOCAL, dirs_exist_ok=True)
    SRC = LOCAL
stats['copy_to_local_s'] = round(time.time() - t, 1)
print('base weights on local disk after', stats['copy_to_local_s'], 's ->', SRC, flush=True)
t = time.time()
# no bitsandbytes: its 8-bit load sat at 0/729 tensors for minutes; fp16 (~16.6 GB) fits across two T4s
te = Qwen2_5_VLForConditionalGeneration.from_pretrained(SRC, subfolder='text_encoder', torch_dtype=DT,
        device_map='auto', max_memory={0: '13GiB', 1: '14GiB'})
print('text encoder loaded in', round(time.time() - t, 1), 's', flush=True)
vae = AutoencoderKLQwenImage.from_pretrained(SRC, subfolder='vae', torch_dtype=DT).to('cuda:0')
pipe = Q.make_pipeline(te, vae, SRC)
stats['stage1_load_s'] = round(time.time() - t, 1); t = time.time()
emb = tuple(x.to('cuda:0') for x in Q.encode(pipe, CAPTION, device='cuda:0'))
emb_x = {k: tuple(x.to('cuda:0') for x in Q.encode(pipe, c, device='cuda:0')) for k, (_, c) in EXTRA.items()}
stats['stage1_encode_s'] = round(time.time() - t, 1); peak('stage1')
del te; Q.unload_text_encoder(pipe, DT); torch.cuda.reset_peak_memory_stats()
print('after unload, GB allocated:', round(torch.cuda.memory_allocated(0) / 1e9, 2)); save(); print(stats)

In [ ]:
from huggingface_hub import hf_hub_download
from diffusers import QwenImageTransformer2DModel, GGUFQuantizationConfig
t = time.time()
tr = QwenImageTransformer2DModel.from_single_file(
    GGUF or hf_hub_download('unsloth/Qwen-Image-Layered-GGUF', 'qwen-image-layered-Q4_K_S.gguf'),
    quantization_config=GGUFQuantizationConfig(compute_dtype=DT), torch_dtype=DT,
    config=SRC, subfolder='transformer').to('cuda:0')
stats['gguf_params_dequantized'] = Q.fix_gguf_non_linear(tr, DT)  # BF16 embedding etc. (see module)
stats['transformer_load_s'] = round(time.time() - t, 1)
print('loaded in', stats['transformer_load_s'], 's; GB allocated', round(torch.cuda.memory_allocated(0) / 1e9, 2), flush=True)


In [ ]:
_BAD = False
def _report(tag, x):
    global _BAD
    x = x.detach().float()
    _BAD = _BAD or not torch.isfinite(x).all().item()
    print(tag, 'nan:', torch.isnan(x).any().item(), 'inf:', torch.isinf(x).any().item(),
          'absmax:', round(x.nan_to_num(0, 0, 0).abs().max().item(), 2), flush=True)
_orig_dl, _orig_vd = Q.decode_latents, pipe.vae.decode
def _dl(p, lat, *a, **k):
    _report('latents', lat); return _orig_dl(p, lat, *a, **k)
def _vd(z, *a, **k):
    _report('vae input', z); o = _orig_vd(z, *a, **k); _report('vae output', o[0]); return o
Q.decode_latents, pipe.vae.decode = _dl, _vd


In [ ]:
import sys, gc
sys.last_traceback = sys.last_value = sys.last_type = None   # a stored OOM traceback keeps GPU tensors alive
def _move(x, d):
    if torch.is_tensor(x):
        return x.to(d, non_blocking=True)
    if isinstance(x, dict):
        return {k: _move(v, d) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return type(x)(_move(v, d) for v in x)
    return x
def _to_own_device(m, args, kwargs):
    d = next(m.parameters()).device
    return _move(args, d), _move(kwargs, d)
def split_across_gpus(model, devices=('cuda:0', 'cuda:1')):
    blocks = model.transformer_blocks
    half = len(blocks) // 2
    for name, child in model.named_children():
        if name != 'transformer_blocks':
            child.to(devices[0])
    for i, b in enumerate(blocks):
        b.to(devices[0] if i < half else devices[1])
    if not getattr(model, '_split_hooks', False):
        for m in list(blocks) + [model.norm_out, model.proj_out]:
            m.register_forward_pre_hook(_to_own_device, with_kwargs=True)
        model._split_hooks = True
    return model


### Mixed precision: fp32 residual stream, fp16 tensor-core matmuls (ComfyUI also lists Qwen-Image as bf16/fp32 only)

In [ ]:
# fp32 residual stream + fp16 tensor-core matmuls: see layerproto/fp16_mixed.py
import gc, sys
sys.last_traceback = sys.last_value = sys.last_type = None
from layerproto.fp16_mixed import apply_fp16_mixed
print('linears patched, params moved to fp32:', apply_fp16_mixed(tr), flush=True)
gc.collect(); torch.cuda.empty_cache(); split_across_gpus(tr)
for _i in range(torch.cuda.device_count()):
    torch.cuda.reset_peak_memory_stats(_i)
Q.decode_latents, pipe.vae.decode = _dl, _vd
try:
    _s7, _st7 = Q.decompose(pipe, tr, img, CAPTION, emb, steps=2, layers=4, resolution=384, true_cfg=1.0)
finally:
    Q.decode_latents, pipe.vae.decode = _orig_dl, _orig_vd
print(_st7, '| peak GB per GPU:', [round(torch.cuda.max_memory_allocated(i) / 1e9, 2) for i in range(torch.cuda.device_count())], flush=True)
Q.strip([img] + _s7).save(f'{OUT}/mixed_smoke_strip.png'); display(Q.strip([img] + _s7))
stats['mixed_smoke'] = _st7; save()
if _BAD:   # find the first module (execution order) whose output is non-finite, 1 step, then stop the run
    class _Found(Exception):
        pass
    def _amax(ts):
        return [round(t.detach().float().nan_to_num(0, 0, 0).abs().max().item(), 1) for t in ts if torch.is_tensor(t) and t.is_floating_point()]
    def _mk(name):
        def _hook(m, i, o):
            for t in (o if isinstance(o, tuple) else (o,)):
                if torch.is_tensor(t) and t.is_floating_point() and not torch.isfinite(t).all():
                    raise _Found(f'{name} ({type(m).__name__}) in {_amax(i)} out {_amax(o if isinstance(o, tuple) else (o,))} dtype {t.dtype}')
        return _hook
    _hs = [m.register_forward_hook(_mk(n)) for n, m in tr.named_modules() if n]
    try:
        split_across_gpus(tr); Q.decompose(pipe, tr, img, CAPTION, emb, steps=1, layers=4, resolution=640)
        stats['first_nonfinite'] = 'none inside the transformer (NaN arises outside it)'
    except _Found as e:
        stats['first_nonfinite'] = str(e)
    finally:
        for h in _hs:
            h.remove()
    print('FIRST NON-FINITE:', stats['first_nonfinite'], flush=True); save()
assert not _BAD, 'NaN/inf still present - stopping before the 50-step run'


In [ ]:

# Run v9: (1) Stable-Layers with Heun via Q.HeunVelocity (the pipeline's Euler step becomes an exact Heun step, so the
# custom-sigma error of FlowMatchHeunDiscreteScheduler in v8 is avoided); (2) 2-layer splits (fewer frames -> faster);
# base model for comparison. All 384 px, CFG 1, both T4s.
import numpy as np, json as _json
from huggingface_hub import hf_hub_download
from layerproto.fp16_mixed import add_lora, remove_lora
cfg = _json.load(open(hf_hub_download('StabilityLabs/Stable-Layers', 'model/adapter_config.json')))
lp = hf_hub_download('StabilityLabs/Stable-Layers', 'model/adapter_model.safetensors')
lscale = cfg.get('lora_alpha', cfg.get('r', 1)) / cfg.get('r', 1)
IMGS = {'forest': (img, emb), **{k: (EXTRA[k][0], emb_x[k]) for k in EXTRA}}
import math
from huggingface_hub import hf_hub_download
from diffusers import FlowMatchEulerDiscreteScheduler
LIGHT_SCHED = dict(base_image_seq_len=256, base_shift=math.log(3), invert_sigmas=False, max_image_seq_len=8192,
                   max_shift=math.log(3), num_train_timesteps=1000, shift=1.0, shift_terminal=None,
                   stochastic_sampling=False, time_shift_type='exponential', use_beta_sigmas=False,
                   use_dynamic_shifting=True, use_exponential_sigmas=False, use_karras_sigmas=False)
orig_sched = pipe.scheduler
lightp = hf_hub_download('lightx2v/Qwen-Image-Lightning', 'Qwen-Image-Lightning-4steps-V2.0-bf16.safetensors')
CONFIGS = [  # (variant, sampler, image, resolution, layers, steps)
    # v10 part 1: Qwen-Image-2.1 story-event composites (night / rain edits, text-to-image scene) re-split with Stable-Layers (Heun)
    ('stable_layers', 'heun', 'q21_night', 384, 5, 4), ('stable_layers', 'heun', 'q21_rain', 384, 5, 4), ('stable_layers', 'heun', 'q21_t2i', 384, 5, 4),
    ('base', 'euler', 'q21_night', 384, 5, 4), ('base', 'euler', 'q21_rain', 384, 5, 4), ('base', 'euler', 'q21_t2i', 384, 5, 4),
    # part 2: Lightning LoRA at 2 and 4 steps, scale sweep, on the fox scene
    ('lightning0.25', 'euler', 'fox_cottage', 384, 5, 2), ('lightning0.5', 'euler', 'fox_cottage', 384, 5, 2),
    ('lightning0.75', 'euler', 'fox_cottage', 384, 5, 2), ('lightning1.0', 'euler', 'fox_cottage', 384, 5, 2),
    ('lightning0.5', 'euler', 'fox_cottage', 384, 5, 4), ('lightning1.0', 'euler', 'fox_cottage', 384, 5, 4),
    ('lightning1.0', 'euler', 'q21_night', 384, 5, 2),
]
stats['sweep'] = []
split_across_gpus(tr)
for variant, sampler, key, res, nl, steps in CONFIGS:
    remove_lora(tr)
    pipe.scheduler = orig_sched
    if variant.startswith('lightning'):
        hit, miss = add_lora(tr, lightp, scale=float(variant[len('lightning'):]))
        stats.setdefault('light_match', [hit, len(miss)])
        pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config(LIGHT_SCHED)
    if variant == 'stable_layers':
        hit, miss = add_lora(tr, lp, scale=lscale)
        stats.setdefault('lora_match', [hit, len(miss)])
    im, e = IMGS[key]
    torch.cuda.synchronize(); t = time.time()
    try:
        out, st = Q.decompose(pipe, tr, im, CAPTION, e, steps=steps, layers=nl, resolution=res, true_cfg=1.0, offload=False, sampler=sampler)
    except Exception as ex:
        print('FAILED', variant, sampler, key, repr(ex)[:300], flush=True); stats['sweep'].append(dict(variant=variant, sampler=sampler, image=key, error=repr(ex)[:300])); save(); continue
    total = round(time.time() - t, 1)
    canvas, info = Q.recompose(out, im.size)
    tag = f'{variant}_{sampler}_{key}_{res}px_{nl}L_{steps}st'
    row = dict(variant=variant, sampler=sampler, image=key, res=res, layers=nl, steps=steps, total_s=total, **st,
               transparent_pct=[i['transparent_pct'] for i in info],
               recomposite_diff=round(float(np.abs(np.array(canvas.convert('RGB').resize(im.size), dtype=int) - np.array(im, dtype=int)).mean()), 2))
    stats['sweep'].append(row); save(); print(row, flush=True)
    Q.strip([im, canvas] + out).save(f'{OUT}/v9_{tag}.png')
remove_lora(tr); pipe.scheduler = orig_sched


## Results